**Clean Kaggle Data**

In [1]:
import pandas as pd
import json
import os
import re
from langdetect import detect, LangDetectException

# Load the data
reviews_path = '../data/kaggle data/game_reviews.csv'
games_path = '../data/kaggle data/gameid.csv'

reviews_df = pd.read_csv(reviews_path)
games_df = pd.read_csv(games_path)

print(f"Reviews: {len(reviews_df):,} rows")
print(f"Games: {len(games_df):,} rows")

# Merge reviews with game names on app_id
merged_df = reviews_df.merge(games_df, left_on='app_id', right_on='appid', how='left')

# Convert is_positive to boolean (True for Positive, False for Negative)
merged_df['is_positive'] = merged_df['is_positive'].str.strip().str.lower() == 'positive'

# ============ DATA CLEANING ============
print("\nCleaning data...")

# 1. Drop rows with missing game names or content
cleaned_df = merged_df.dropna(subset=['name', 'content'])
print(f"After dropping nulls: {len(cleaned_df):,}")

# 2. Convert content to string and strip whitespace
cleaned_df = cleaned_df.copy()
cleaned_df['content'] = cleaned_df['content'].astype(str).str.strip()

# 3. Remove empty reviews
cleaned_df = cleaned_df[cleaned_df['content'].str.len() > 0]
print(f"After removing empty: {len(cleaned_df):,}")

# 4. Remove reviews without at least one actual word (must have letters)
def has_valid_word(text):
    """Check if text contains at least one word with 2+ letters"""
    words = re.findall(r'[a-zA-Z]{2,}', text)
    return len(words) >= 1

cleaned_df = cleaned_df[cleaned_df['content'].apply(has_valid_word)]
print(f"After requiring valid words: {len(cleaned_df):,}")

# 5. Filter to English-only reviews
def is_english(text):
    """Detect if text is English using langdetect"""
    try:
        return detect(text) == 'en'
    except LangDetectException:
        return False

print("Filtering non-English reviews (this may take a few minutes)...")
cleaned_df = cleaned_df[cleaned_df['content'].apply(is_english)]
print(f"After English filter: {len(cleaned_df):,}")

# 6. Remove very short reviews (less than 20 characters after cleaning)
cleaned_df = cleaned_df[cleaned_df['content'].str.len() >= 20]
print(f"After min length filter: {len(cleaned_df):,}")

# Select only the columns we need
final_df = cleaned_df[['name', 'content', 'is_positive']].copy()
final_df.columns = ['game_name', 'review_content', 'is_positive']

print(f"\n=== Final Dataset ===")
print(f"Total reviews: {len(final_df):,}")
print(f"Positive reviews: {final_df['is_positive'].sum():,}")
print(f"Negative reviews: {(~final_df['is_positive']).sum():,}")

# Convert to list of dictionaries for JSON
data_list = final_df.to_dict(orient='records')

# Save to JSON
output_dir = '../data/cleaned data'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'kaggle_data.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(data_list, f, ensure_ascii=False, indent=2)

print(f"\nSaved to {output_path}")
print(f"\nSample entries:")
for i in range(min(3, len(data_list))):
    print(f"\n--- Review {i+1} ---")
    print(json.dumps(data_list[i], indent=2, ensure_ascii=False))

Reviews: 201,151 rows
Games: 1,000 rows

Cleaning data...
After dropping nulls: 191,655
After removing empty: 191,600
After requiring valid words: 183,466
Filtering non-English reviews (this may take a few minutes)...
After English filter: 135,244
After min length filter: 117,458

=== Final Dataset ===
Total reviews: 117,458
Positive reviews: 55,789
Negative reviews: 61,669

Saved to ../data/cleaned data/kaggle_data.json

Sample entries:

--- Review 1 ---
{
  "game_name": "Alien Swarm",
  "review_content": "my friend fucking glitched out of the map. dude why",
  "is_positive": false
}

--- Review 2 ---
{
  "game_name": "Alien Swarm",
  "review_content": "nobody is playing this game",
  "is_positive": false
}

--- Review 3 ---
{
  "game_name": "Alien Swarm",
  "review_content": "this game isn't very fun it runs like ass and the community is ready to kick you as soon as you make 1 mistake so don't even waste your time unless you want the silly tf2 hat.",
  "is_positive": false
}


**Clean Mendeley Data**